# Day 06 — Pivot Tables + Reshaping
**Estimated time:** 60–75 min
**Dataset:** `cost_center_actuals.csv`

## Learning Objectives
- Build pivot tables with `pivot_table()` and multiple aggregation functions
- Unpivot wide/BW-extract formats to tidy long format with `melt()`
- Use `stack()`/`unstack()` to reshape multi-level DataFrames

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
cc = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
cc["ACTUAL_AMT"] = pd.to_numeric(cc["ACTUAL_AMT"], errors="coerce")
cc["PLAN_AMT"]   = pd.to_numeric(cc["PLAN_AMT"],   errors="coerce")
cc["PERIOD"]     = pd.to_numeric(cc["PERIOD"],      errors="coerce")
cc["GJAHR"]      = pd.to_numeric(cc["GJAHR"],       errors="coerce")

print(cc.shape)
display(cc.head(8))
print(cc.dtypes)

In [ ]:
# -- 1. Basic pivot_table --
# Actual spend by cost center x fiscal period
pivot_actual = pd.pivot_table(
    cc,
    values="ACTUAL_AMT",
    index="KOSTL",          # rows
    columns="PERIOD",       # columns
    aggfunc="sum",
    fill_value=0
)
print("Shape:", pivot_actual.shape)
display(pivot_actual.head(10))

In [ ]:
# -- 2. Multi-aggfunc pivot --
# Both actual and plan in one pivot
pivot_both = pd.pivot_table(
    cc[cc["GJAHR"] == cc["GJAHR"].max()],   # latest fiscal year
    values=["ACTUAL_AMT","PLAN_AMT"],
    index="KOSTL",
    columns="PERIOD",
    aggfunc="sum",
    fill_value=0
)
display(pivot_both.head(8))

In [ ]:
# -- 3. Calculate variance and flag > 10% overspend --
# Flatten multi-level columns for easier manipulation
pivot_flat = pd.pivot_table(
    cc[cc["GJAHR"] == cc["GJAHR"].max()],
    values=["ACTUAL_AMT","PLAN_AMT"],
    index="KOSTL",
    aggfunc="sum",
    fill_value=0
).reset_index()

pivot_flat.columns = ["KOSTL","actual","plan"]
pivot_flat["variance"]      = pivot_flat["actual"] - pivot_flat["plan"]
pivot_flat["variance_pct"]  = (pivot_flat["variance"] / pivot_flat["plan"].replace(0, np.nan) * 100).round(2)
pivot_flat["overspend_flag"] = pivot_flat["variance_pct"] > 10

overspend = pivot_flat[pivot_flat["overspend_flag"]].sort_values("variance_pct", ascending=False)
print(f"Cost centers with >10% overspend: {len(overspend)}")
display(overspend)

In [ ]:
# -- 4. melt() — wide BW extract -> tidy long format --
# Simulate a wide BW extract: one row per cost center, periods as columns
wide_extract = pivot_actual.reset_index()
print("Wide format sample:")
display(wide_extract.iloc[:3, :6])

# Melt to long (tidy) format
long_format = wide_extract.melt(
    id_vars=["KOSTL"],
    var_name="PERIOD",
    value_name="ACTUAL_AMT"
)
print(f"\nLong format: {len(long_format):,} rows")
display(long_format.head(10))

In [ ]:
# -- 5. stack() and unstack() --
# stack: moves column level -> row level (wide -> long)
# unstack: moves row level -> column level (long -> wide)

stacked = pivot_both.stack(level=0, future_stack=True)
print("After stack():")
display(stacked.head(10))

unstacked = stacked.unstack("PERIOD")
print("\nAfter unstack():")
display(unstacked.head(5))

In [ ]:
# -- 6. crosstab for frequency analysis --
# Count combinations of cost center (KOSTL) and cost element group (KSTAR prefix)
cc["KSTAR_GRP"] = cc["KSTAR"].astype(str).str[:2]   # leading 2 chars = group

crosstab = pd.crosstab(cc["KSTAR_GRP"], cc["GJAHR"], margins=True)
print("Cost element group x Fiscal year frequency:")
display(crosstab)

---
## Daily Prompt

**Build a pivot showing actual vs plan spend by cost center and fiscal period for the most recent fiscal year. Highlight (flag) any cell where actual exceeds plan by more than 10%.**

```python
# YOUR SOLUTION
latest_year = cc["GJAHR"].max()
year_data = cc[cc["GJAHR"] == latest_year]

pivot_a = pd.pivot_table(year_data, values="ACTUAL_AMT", index="KOSTL",
                          columns="PERIOD", aggfunc="sum", fill_value=0)
pivot_p = pd.pivot_table(year_data, values="PLAN_AMT", index="KOSTL",
                          columns="PERIOD", aggfunc="sum", fill_value=0)

# YOUR SOLUTION: calculate variance_pct and create overspend_mask
# variance_pct = (pivot_a - pivot_p) / pivot_p.replace(0, np.nan) * 100
# overspend_mask = variance_pct > 10
```

> **Hint:** Style the DataFrame with:
> `pivot_a.style.apply(lambda df: np.where(overspend_mask, "background-color: salmon", ""), axis=None)`